# Lending Decision Model: PD → Expected NPV → Approval Cut-off

**Question:** who should we lend to? Approving everyone loses money on defaults; rejecting too many loses profitable customers.
This notebook:

1. Uses **SQL** to profile default rates by segment
2. Runs **hypothesis tests** to check which differences are real
3. Builds a **probability-of-default (PD)** model
4. Turns PD into an **expected NPV per loan** and picks the approval policy that maximises portfolio NPV

Data: German Credit (1,000 loans, 30% default rate). Amounts are in Deutsche Mark (DM).

In [1]:
import sqlite3
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv("german_credit_full_1000.csv")
df["default"] = 1 - df["credit_risk"]   # credit_risk 1 = good, so default = 1 when credit_risk = 0
print(df.shape, "| default rate:", df["default"].mean())

(1000, 22) | default rate: 0.3


In [2]:
con = sqlite3.connect(":memory:")
df.to_sql("loans", con, index=False)

query = '''
SELECT status AS checking_account,
       COUNT(*)                 AS loans,
       ROUND(AVG("default"), 3) AS default_rate,
       ROUND(AVG(amount), 0)    AS avg_amount
FROM loans
GROUP BY status
ORDER BY default_rate DESC
'''
seg = pd.read_sql(query, con)
seg

,checking_account,loans,default_rate,avg_amount
0,... < 100 DM,274,0.493,3175.0
1,0 <= ... < 200 DM,269,0.390,3828.0
2,... >= 200 DM / salary for at least 1 year,63,0.222,2178.0
3,no checking account,394,0.117,3133.0


## 2. Hypothesis tests

- **Chi-square:** is default rate independent of checking-account status? (H0: no difference)
- **Welch t-test:** do defaulters take longer loans? (H0: same mean duration)

In [3]:
table = pd.crosstab(df["status"], df["default"])
chi2, p_chi, dof, _ = stats.chi2_contingency(table)
print(f"Chi-square = {chi2:.1f}, dof = {dof}, p = {p_chi:.2e}")

dur_bad  = df.loc[df["default"] == 1, "duration"]
dur_good = df.loc[df["default"] == 0, "duration"]
t, p_t = stats.ttest_ind(dur_bad, dur_good, equal_var=False)
print(f"Mean duration: defaulters {dur_bad.mean():.1f} vs non-defaulters {dur_good.mean():.1f} months")
print(f"Welch t = {t:.2f}, p = {p_t:.2e}")

Chi-square = 123.7, dof = 3, p = 1.22e-26
Mean duration: defaulters 24.9 vs non-defaulters 19.2 months
Welch t = 6.47, p = 2.40e-10


In [4]:
X = pd.get_dummies(df.drop(columns=["credit_risk", "default"]), drop_first=True)
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42)

model = GradientBoostingClassifier(random_state=42).fit(X_train, y_train)
pd_test = model.predict_proba(X_test)[:, 1]
print("Test AUC:", round(roc_auc_score(y_test, pd_test), 3), "| test loans:", len(y_test))

Test AUC: 0.794 | test loans: 300


## 4. From PD to expected NPV

Assumptions (illustrative, stated so they can be challenged):

| Assumption | Value |
|---|---|
| Customer APR | 15% |
| Funding cost / discount rate | 5% a year |
| Servicing cost | 2% of loan amount |
| Defaulters | pay for the first half of the term, then stop |
| Recovery on default | 40% of outstanding balance (LGD 60%) |

Each loan is an amortising monthly loan over its actual term.

In [5]:
APR, FUND, OPEX, RECOVERY = 0.15, 0.05, 0.02, 0.40

def loan_npvs(amount, months):
    r, d = APR / 12, FUND / 12
    pay = amount * r / (1 - (1 + r) ** -months)          # monthly instalment
    t = np.arange(1, months + 1)
    disc = (1 + d) ** -t
    npv_good = (pay * disc).sum() - amount - OPEX * amount

    k = months // 2                                        # months paid before default
    bal_k = amount * (1 + r) ** k - pay * ((1 + r) ** k - 1) / r
    npv_bad = (pay * disc[:k]).sum() + RECOVERY * bal_k * (1 + d) ** -(k + 1) - amount - OPEX * amount
    return npv_good, npv_bad

test = X_test[[]].copy()
test["amount"]   = df.loc[X_test.index, "amount"]
test["duration"] = df.loc[X_test.index, "duration"]
test["default"]  = y_test
test["pd"]       = pd_test
test[["npv_good", "npv_bad"]] = [loan_npvs(a, m) for a, m in zip(test["amount"], test["duration"])]

test["expected_npv"] = (1 - test["pd"]) * test["npv_good"] + test["pd"] * test["npv_bad"]
test["realised_npv"] = np.where(test["default"] == 1, test["npv_bad"], test["npv_good"])
test.head()

,amount,duration,default,pd,npv_good,npv_bad,expected_npv,realised_npv
115,3578,48,0,0.023033,674.429227,-639.393965,644.168199,674.429227
346,882,13,0,0.225197,33.994140,-271.078624,-34.707270,33.994140
328,4473,36,0,0.460041,611.159390,-960.209454,-111.734585,611.159390
974,2831,30,0,0.102125,313.829238,-658.020817,214.578589,313.829238
587,1289,12,0,0.157024,44.247998,-367.057909,-20.336796,44.247998


## 5. Approval policy

Compare three policies on the **held-out test loans**, scoring each on what actually happened (realised NPV):

- **Approve all**
- **Flat PD cut-off** (best single threshold)
- **Expected-NPV rule:** approve only if expected NPV > 0. This lets the cut-off flex with loan size and term.

In [6]:
def summarise(name, approve):
    a = test[approve]
    return {"policy": name,
            "approval_rate": round(approve.mean(), 3),
            "bad_rate": round(a["default"].mean(), 3),
            "portfolio_npv": round(a["realised_npv"].sum())}

rows = [summarise("Approve all", np.ones(len(test), bool))]

grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
cut_npv = [test.loc[test["pd"] <= c, "realised_npv"].sum() for c in grid]
best_cut = grid[int(np.argmax(cut_npv))]
rows.append(summarise(f"PD cut-off <= {best_cut}", test["pd"] <= best_cut))
rows.append(summarise("Expected NPV > 0", test["expected_npv"] > 0))

policies = pd.DataFrame(rows)
policies

,policy,approval_rate,bad_rate,portfolio_npv
0,Approve all,1.000,0.300,-35354
1,PD cut-off <= 0.26,0.567,0.124,11306
2,Expected NPV > 0,0.457,0.190,9931


**Caveat:** the flat cut-off is chosen on the same test loans it is scored on, so it is optimistic.
The expected-NPV rule uses no outcome information, so it is the fair comparison.

In [7]:
base = policies.iloc[0]
rule = policies.iloc[2]
print(f"Expected-NPV rule vs approve-all: NPV {base.portfolio_npv:,} -> {rule.portfolio_npv:,} DM; "
      f"approvals {base.approval_rate:.0%} -> {rule.approval_rate:.0%}; "
      f"bad rate {base.bad_rate:.0%} -> {rule.bad_rate:.0%}")

Expected-NPV rule vs approve-all: NPV -35,354 -> 9,931 DM; approvals 100% -> 46%; bad rate 30% -> 19%
